# Backend de inferencia ASR → texto/glosas/video en LSEC

Este notebook está preparado para **Google Colab** y tiene como finalidad levantar el servidor backend de inferencia mediante **FastAPI + Uvicorn**.

El flujo esperado es:

1. Instalar Conda en Colab mediante `condacolab`.
2. Descargar el respaldo del backend desde Google Drive.
3. Descomprimir el backend y verificar que exista el entorno `wf`.
4. Verificar que exista el archivo `server_fastapi_raw_video.py`.
5. Descargar recursos auxiliares requeridos por el backend, como WordNet.
6. Liberar el puerto 8000 si ya estaba ocupado.
7. Levantar el servidor FastAPI.
8. Exponer el servidor mediante Cloudflare Tunnel.

> Este notebook **no entrena el modelo**. Solo prepara y ejecuta el backend de inferencia.


## 0. Requisito previo: respaldo del backend

Este notebook asume que ya existe un archivo comprimido en Google Drive con los componentes necesarios para ejecutar el backend. En el caso de este proyecto, el respaldo debe contener, como mínimo:

```text
/content/whisper-flamingo/
└── server_fastapi_raw_video.py

/usr/local/envs/wf/
└── bin/python
```

El directorio `/content/whisper-flamingo` contiene el código del backend y el entorno `/usr/local/envs/wf` contiene las dependencias necesarias para ejecutarlo.

Si el respaldo tiene otra estructura, modifica las variables `BACKEND_DIR` y `ENV_PYTHON` en la celda de configuración.


## 1. Instalar Conda en Colab

Ejecuta esta celda solo una vez al inicio. `condacolab.install()` puede reiniciar el entorno de ejecución. Si Colab reinicia el runtime, vuelve a ejecutar el notebook desde la siguiente celda.


In [1]:
# ============================================================
# Instalación de Conda en Google Colab
# ============================================================
# Esta celda prepara Colab para utilizar entornos Conda.
# En este proyecto se usa el entorno /usr/local/envs/wf,
# que debe venir incluido en el respaldo del backend.

!pip -q install condacolab

import condacolab
condacolab.install()


⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:10
🔁 Restarting kernel...


## 2. Configuración general del backend

Modifica estas variables si cambias el archivo de Google Drive, el nombre del respaldo, el puerto o la ruta del backend.


In [31]:
from pathlib import Path

# ============================================================
# Parámetros principales del backend
# ============================================================

# ID del archivo en Google Drive.
# Este valor proviene del notebook original. Cámbialo si subes otro respaldo.
GDRIVE_FILE_ID = "1oiVpmsmlEd6Uw1t3fPa2W_v0gPt-xx6G"

# Carpeta raíz donde se descomprimirá el respaldo.
# Se usa "/" porque el respaldo puede contener rutas como:
# /content/whisper-flamingo y /usr/local/envs/wf.
EXTRACT_ROOT = Path("/")

# Ruta esperada del código del backend.
BACKEND_DIR = Path("/content/whisper-flamingo")

# Python del entorno Conda usado por el backend.
ENV_PYTHON = Path("/usr/local/envs/wf/bin/python")

# Archivo FastAPI que contiene la aplicación.
# server_fastapi_raw_video.py debe contener una variable llamada "app".
SERVER_APP = "server_fastapi_raw_video:app"

# Puerto local del backend.
PORT = 8000
HOST = "0.0.0.0"

# Archivos auxiliares para controlar el proceso.
FASTAPI_PID = Path("/content/fastapi.pid")
FASTAPI_LOG = Path("/content/fastapi.log")

CLOUDFLARED_PID = Path("/content/cloudflared.pid")
CLOUDFLARED_LOG = Path("/content/cloudflared.log")

print("Configuración cargada:")
print("Archivo Drive ID:", GDRIVE_FILE_ID)
print("Archivo local:", ARCHIVE_PATH)
print("Ruta backend:", BACKEND_DIR)
print("Python entorno wf:", ENV_PYTHON)
print("Servidor:", SERVER_APP)
print("Puerto:", PORT)


Configuración cargada:
Archivo Drive ID: 1oiVpmsmlEd6Uw1t3fPa2W_v0gPt-xx6G
Archivo local: /content/backend_inferencia.zip
Ruta backend: /content/whisper-flamingo
Python entorno wf: /usr/local/envs/wf/bin/python
Servidor: server_fastapi_raw_video:app
Puerto: 8000


## 3. Descargar el respaldo del backend desde Google Drive

Esta celda descarga el archivo comprimido del backend. El archivo se guarda en `/content/backend_inferencia.zip`.

Si el archivo está en tu Google Drive personal y no es público, primero deberás montarlo o hacerlo accesible con enlace compartido.


In [10]:
import subprocess
from pathlib import Path

# ============================================================
# Descarga del respaldo del backend
# ============================================================
# Se instala gdown para descargar archivos desde Google Drive.
# El ID del archivo se definió en la celda de configuración.

subprocess.run(["python", "-m", "pip", "install", "-q", "gdown"], check=True)

if ARCHIVE_PATH.exists():
    ARCHIVE_PATH.unlink()

cmd = ["gdown", GDRIVE_FILE_ID, "-O", str(ARCHIVE_PATH)]
print("Ejecutando:", " ".join(cmd))
subprocess.run(cmd, check=True)

if not ARCHIVE_PATH.exists() or ARCHIVE_PATH.stat().st_size == 0:
    raise FileNotFoundError("No se descargó correctamente el respaldo del backend.")

print("Respaldo descargado correctamente:")
print(ARCHIVE_PATH)
print("Tamaño:", round(ARCHIVE_PATH.stat().st_size / (1024 * 1024), 2), "MB")


Ejecutando: gdown 1oiVpmsmlEd6Uw1t3fPa2W_v0gPt-xx6G -O /content/backend_inferencia.zip
Respaldo descargado correctamente:
/content/backend_inferencia.zip
Tamaño: 13878.04 MB


## 4. Descomprimir el respaldo del backend



In [34]:
%%bash
# Extraemos los archivos desde Drive hacia la raíz (/) para que caigan en su lugar correcto
tar -xzvf /content/backend_inferencia.zip -C /

Process is interrupted.


## 5. Verificar estructura del backend

Antes de levantar el servidor, se verifica que existan:

- el directorio del backend;
- el archivo `server_fastapi_raw_video.py`;
- el entorno Conda `wf`;
- el ejecutable `/usr/local/envs/wf/bin/python`.

Si alguna ruta no existe, el servidor no podrá iniciar.


In [15]:
from pathlib import Path
import subprocess

# ============================================================
# Verificación de rutas requeridas
# ============================================================

# Si BACKEND_DIR no existe, se intenta localizar automáticamente
# el archivo server_fastapi_raw_video.py dentro de /content.
if not BACKEND_DIR.exists():
    candidates = list(Path("/content").rglob("server_fastapi_raw_video.py"))
    if candidates:
        BACKEND_DIR = candidates[0].parent
        print("BACKEND_DIR actualizado automáticamente a:", BACKEND_DIR)

server_file = BACKEND_DIR / "server_fastapi_raw_video.py"

checks = {
    "Directorio backend": BACKEND_DIR,
    "Archivo server_fastapi_raw_video.py": server_file,
    "Python del entorno wf": ENV_PYTHON,
}

for name, path in checks.items():
    print(f"{name}: {path}")
    if not path.exists():
        raise FileNotFoundError(f"No existe: {path}")

print("\nEstructura mínima encontrada correctamente.")

# Mostrar versión de Python usada por el backend.
subprocess.run([str(ENV_PYTHON), "--version"], check=True)


Directorio backend: /content/whisper-flamingo
Archivo server_fastapi_raw_video.py: /content/whisper-flamingo/server_fastapi_raw_video.py
Python del entorno wf: /usr/local/envs/wf/bin/python

Estructura mínima encontrada correctamente.


CompletedProcess(args=['/usr/local/envs/wf/bin/python', '--version'], returncode=0)

## 6. Descargar recursos auxiliares de NLTK

El backend utiliza recursos lingüísticos para algunas fases de procesamiento de texto. Por eso se descarga `wordnet` dentro del entorno `wf`.

Si tu versión del backend no usa NLTK, esta celda no afecta negativamente.


In [16]:
import subprocess

# ============================================================
# Descarga de recursos NLTK en el entorno wf
# ============================================================
# Se ejecuta con /usr/local/envs/wf/bin/python para asegurar que
# los recursos queden disponibles en el mismo entorno del backend.

nltk_cmd = (
    "import nltk; "
    "nltk.download('wordnet'); "
    "nltk.download('omw-1.4')"
)

subprocess.run([str(ENV_PYTHON), "-c", nltk_cmd], check=True)

print("Recursos NLTK descargados en el entorno wf.")


Recursos NLTK descargados en el entorno wf.


## 7. Liberar puerto y detener servidores anteriores

Antes de iniciar FastAPI se detiene cualquier proceso anterior asociado al puerto 8000 o al archivo `fastapi.pid`.

Esto evita errores del tipo:

```text
Address already in use
```


In [17]:
import os
import signal
import subprocess
from pathlib import Path

# ============================================================
# Limpieza de procesos anteriores
# ============================================================

def kill_pid_file(pid_file: Path, name: str):
    if pid_file.exists():
        try:
            pid = int(pid_file.read_text().strip())
            os.kill(pid, signal.SIGTERM)
            print(f"{name} detenido. PID:", pid)
        except ProcessLookupError:
            print(f"{name}: el proceso ya no existía.")
        except Exception as e:
            print(f"No se pudo detener {name} desde {pid_file}: {e}")
        finally:
            pid_file.unlink(missing_ok=True)


# Detener servidor FastAPI anterior si existe.
kill_pid_file(FASTAPI_PID, "FastAPI")

# Liberar puerto 8000 si quedó ocupado por otro proceso.
subprocess.run(
    f"fuser -k {PORT}/tcp || true",
    shell=True,
    check=False
)

# Eliminar log anterior para no mezclar ejecuciones.
if FASTAPI_LOG.exists():
    FASTAPI_LOG.unlink()

print("Puerto liberado y logs anteriores eliminados.")


Puerto liberado y logs anteriores eliminados.


## 8. Levantar el servidor FastAPI

Esta celda inicia el backend con Uvicorn usando el Python del entorno `wf`.

El servidor se ejecuta en segundo plano y sus mensajes se guardan en:

```text
/content/fastapi.log
```

La documentación automática de la API queda disponible localmente en:

```text
http://localhost:8000/docs
```


In [18]:
import subprocess
import time
from pathlib import Path

# ============================================================
# Inicio del servidor FastAPI
# ============================================================

cmd = [
    str(ENV_PYTHON),
    "-m",
    "uvicorn",
    SERVER_APP,
    "--host",
    HOST,
    "--port",
    str(PORT),
]

print("Iniciando servidor:")
print(" ".join(cmd))
print("Directorio de ejecución:", BACKEND_DIR)

log_file = open(FASTAPI_LOG, "w", encoding="utf-8")

process = subprocess.Popen(
    cmd,
    cwd=str(BACKEND_DIR),
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

FASTAPI_PID.write_text(str(process.pid))

print("Servidor iniciado con PID:", process.pid)
print("Esperando 10 segundos para revisar logs...")
time.sleep(10)

log_file.flush()
log_file.close()

if FASTAPI_LOG.exists():
    print("\nÚltimas líneas del log:")
    print("-" * 80)
    print(FASTAPI_LOG.read_text(encoding="utf-8", errors="replace")[-5000:])
    print("-" * 80)

print("\nSi aparece 'Uvicorn running on http://0.0.0.0:8000', el backend está activo.")


Iniciando servidor:
/usr/local/envs/wf/bin/python -m uvicorn server_fastapi_raw_video:app --host 0.0.0.0 --port 8000
Directorio de ejecución: /content/whisper-flamingo
Servidor iniciado con PID: 6017
Esperando 10 segundos para revisar logs...

Últimas líneas del log:
--------------------------------------------------------------------------------
INFO:     Started server process [6017]
INFO:     Waiting for application startup.
[INFO] Python backend: /usr/local/envs/wf/bin/python
[INFO] Python preprocess: /usr/local/envs/wf/bin/python
[INFO] PREPROCESS_AUDIO_SCRIPT: /content/whisper-flamingo/preprocess_single_audio.py
[INFO] Cargando modelo ASR...

--------------------------------------------------------------------------------

Si aparece 'Uvicorn running on http://0.0.0.0:8000', el backend está activo.


## 9. Consultar logs del servidor

Usa esta celda cada vez que quieras revisar si el servidor cargó correctamente el modelo o si hubo algún error durante la inferencia.


In [29]:
# ============================================================
# Mostrar las últimas líneas del log del backend
# ============================================================

!tail -n 100 /content/fastapi.log


INFO:     Started server process [6017]
INFO:     Waiting for application startup.
[INFO] Python backend: /usr/local/envs/wf/bin/python
[INFO] Python preprocess: /usr/local/envs/wf/bin/python
[INFO] PREPROCESS_AUDIO_SCRIPT: /content/whisper-flamingo/preprocess_single_audio.py
[INFO] Cargando modelo ASR...


## 10. Verificar que FastAPI responda localmente

Esta celda consulta la documentación de FastAPI. Si responde con código `200`, el servidor está funcionando.

Si falla, revisa `/content/fastapi.log`.


In [ ]:
# ============================================================
# Verificación local del servidor
# ============================================================

!curl -I http://localhost:8000/docs || true


## 11. Exponer el backend con Cloudflare Tunnel

Colab no expone directamente el puerto 8000 hacia Internet. Para que una aplicación externa, como una app móvil Flutter, pueda comunicarse con el backend, se crea un túnel temporal con `cloudflared`.

El enlace público tendrá una forma similar a:

```text
https://xxxx.trycloudflare.com
```

Ese enlace cambia cada vez que se reinicia el túnel.


In [ ]:
%%bash
set -euo pipefail

# ============================================================
# Iniciar Cloudflare Tunnel hacia http://localhost:8000
# ============================================================

# Detener túnel anterior si existe.
if [ -f /content/cloudflared.pid ]; then
  kill $(cat /content/cloudflared.pid) || true
  rm -f /content/cloudflared.pid
fi

# Instalar cloudflared si no existe.
if ! command -v cloudflared >/dev/null 2>&1; then
  wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
  chmod +x /usr/local/bin/cloudflared
fi

# Limpiar log anterior.
rm -f /content/cloudflared.log

# Crear túnel.
nohup cloudflared tunnel --url http://localhost:8000 > /content/cloudflared.log 2>&1 &

echo $! > /content/cloudflared.pid

sleep 8

echo "Log de Cloudflare:"
cat /content/cloudflared.log

echo ""
echo "URL pública detectada:"
grep -oE "https://[-a-zA-Z0-9.]+trycloudflare.com" /content/cloudflared.log | head -1 || true


## 12. Revisar URL pública del túnel

Si la celda anterior no mostró claramente el enlace público, ejecuta esta celda.


In [ ]:
# ============================================================
# Mostrar URL pública de Cloudflare Tunnel
# ============================================================

!grep -oE "https://[-a-zA-Z0-9.]+trycloudflare.com" /content/cloudflared.log | head -1 || true


## 13. Probar el endpoint desde Colab

Primero abre la documentación de FastAPI en:

```text
http://localhost:8000/docs
```

Ahí podrás ver el nombre exacto del endpoint. En este proyecto suele usarse un endpoint para subir video y obtener transcripción/glosas/video final.

Si tu endpoint se llama, por ejemplo, `/transcribe_video_to_gloss`, puedes probarlo con `curl` adaptando la ruta del video.


In [ ]:
# ============================================================
# Ejemplo de prueba del backend
# ============================================================
# Cambia VIDEO_TEST por la ruta de un video real.
# También cambia el endpoint si en /docs aparece con otro nombre.

VIDEO_TEST = "/content/video_prueba.mp4"
ENDPOINT = "http://localhost:8000/transcribe_video_to_gloss"

print("Ejemplo de comando curl:")
print(
    f'curl -X POST "{ENDPOINT}" '
    f'-F "file=@{VIDEO_TEST}"'
)

# Descomenta estas líneas solo si ya tienes un video de prueba.
# !curl -X POST "$ENDPOINT" -F "file=@/content/video_prueba.mp4"


## 14. Detener backend y túnel

Ejecuta esta celda cuando termines de usar el backend. Esto libera recursos en Colab.


In [ ]:
import os
import signal
from pathlib import Path

# ============================================================
# Detener FastAPI y Cloudflare Tunnel
# ============================================================

def stop_process(pid_file: Path, name: str):
    if pid_file.exists():
        try:
            pid = int(pid_file.read_text().strip())
            os.kill(pid, signal.SIGTERM)
            print(f"{name} detenido. PID:", pid)
        except ProcessLookupError:
            print(f"{name}: el proceso ya no estaba activo.")
        except Exception as e:
            print(f"No se pudo detener {name}: {e}")
        finally:
            pid_file.unlink(missing_ok=True)
    else:
        print(f"No hay PID registrado para {name}.")

stop_process(FASTAPI_PID, "FastAPI")
stop_process(CLOUDFLARED_PID, "Cloudflare Tunnel")

print("Procesos detenidos.")


## Notas de reproducibilidad

Para que otro usuario pueda reproducir el levantamiento del backend, debe documentarse:

- ID o ubicación del respaldo del backend.
- Versión del notebook.
- Ruta esperada del backend: `/content/whisper-flamingo`.
- Ruta del entorno: `/usr/local/envs/wf`.
- Archivo de servidor: `server_fastapi_raw_video.py`.
- Puerto local: `8000`.
- Nombre del endpoint de inferencia, verificable en `/docs`.
- Ruta del checkpoint usado por el backend, configurada dentro del archivo del servidor o en los archivos de configuración correspondientes.

Si el backend falla al iniciar, revisar siempre:

```text
/content/fastapi.log
```

Si el túnel no entrega URL pública, revisar:

```text
/content/cloudflared.log
```
